# E91 Entanglement-Based QKD (PennyLane)

Alice and Bob share Bell pairs and measure in random bases.
Matching bases form the sifted key.

In [ ]:
import random
import pennylane as qml
import numpy as np

NUM_PAIRS = 20
dev = qml.device("default.qubit", wires=2, shots=1)

## Bell pair + measurement circuit

In [ ]:
@qml.qnode(dev)
def e91_measure(alice_basis, bob_basis):
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    if alice_basis == 1:
        qml.RY(3 * np.pi / 4, wires=0)
    elif alice_basis == 2:
        qml.RY(np.pi / 4, wires=0)
    if bob_basis == 1:
        qml.RY(3 * np.pi / 4, wires=1)
    elif bob_basis == 2:
        qml.RY(np.pi / 4, wires=1)
    return qml.sample(wires=[0, 1])

print(qml.draw(e91_measure)(0, 0))

In [ ]:
alice_bases = [random.choice([0, 1, 2]) for _ in range(NUM_PAIRS)]
bob_bases = [random.choice([0, 1, 2]) for _ in range(NUM_PAIRS)]

key_a, key_b = [], []
for i in range(NUM_PAIRS):
    sample = e91_measure(alice_bases[i], bob_bases[i])
    if alice_bases[i] == bob_bases[i]:
        key_a.append(int(sample[0]))
        key_b.append(int(sample[1]))

print(f"Key A: {key_a}")
print(f"Key B: {key_b}")
check = random.sample(range(len(key_a)), min(4, len(key_a)))
errors = sum(1 for i in check if key_a[i] != key_b[i])
print(f"QBER: {errors}/{len(check)} = {errors / len(check):.2%}")